This book is used to visually show which planning area and subzone of singapore does not have demographic data according to 2010, 2015 and 2020 census data

In [18]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import shape
from pathlib import Path
import os
from bs4 import BeautifulSoup
import re

In [19]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [20]:
def read_gpkg_file(filepath: str):
    """
    Returns a geospatial dataframe of the file specified in the filepath
    Returns None if the filepath given does not exist
    """
    if filepath.exists():
        geospatial_df = gpd.read_file(filepath)
    else:
        print(f"filepath given {filepath} does not exists. Aborting")
        return None

    print(geospatial_df["Description"][1])
    return geospatial_df

In [21]:
def extract_subzone_name(description_html: str):
    """
    Parses the HTML-like string in the Description column to extract 
    the value associated with the 'SUBZONE_N' key.
    
    Args:
        description_html (str): The string content from the 'Description' column 
                                (containing the HTML table).
        
    Returns:
        str: The extracted subzone name (e.g., 'BUKIT MERAH') or None if not found.
    """
    # 1. Handle NaN/Null values
    if pd.isna(description_html) or not isinstance(description_html, str):
        return None
        
    try:
        # 2. Use BeautifulSoup to parse the table structure
        soup = BeautifulSoup(description_html, 'html.parser')
        
        # 3. Find all table rows (<tr>)
        rows = soup.find_all('tr')
        
        # 4. Iterate through the rows to find the one containing 'SUBZONE_N'
        for row in rows:
            # Look for the header cell (<th>) with the exact text 'SUBZONE_N'
            header_cell = row.find('th', string='SUBZONE_N')
            
            if header_cell:
                # The value is in the corresponding data cell (<td>) in the same row
                subzone_name_tag = row.find('td')
                
                if subzone_name_tag:
                    return subzone_name_tag.text.strip()
                    
        return None # Return None if the key 'SUBZONE_N' is not found
        
    except Exception as e:
        # Catch any parsing errors
        print(f"Error parsing description for a feature: {e}")
        return None

In [22]:
subzone_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/masterplan_2019/MasterPlan2019SubzoneBoundary.gpkg")
subzone_df = read_gpkg_file(subzone_filepath)
if subzone_df is not None:
    display(subzone_df.head())

subzone_df["SUBZONE_N"] = subzone_df["Description"].apply(extract_subzone_name)
save_to_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/masterplan_2019/subzone_boundary_2019.gpkg")

subzone_df.to_file(
    save_to_filepath,
    layer = "subzone",
    driver = "GPKG",
    mode = "w" # Use 'w' (write mode) to overwrite the existing layer
)

<center><table><tr><th colspan='2' align='center'><em>Attributes</em></th></tr><tr bgcolor="#E3E3F3"> <th>SUBZONE_NO</th> <td>2</td> </tr><tr bgcolor=""> <th>SUBZONE_N</th> <td>BUKIT MERAH</td> </tr><tr bgcolor="#E3E3F3"> <th>SUBZONE_C</th> <td>BMSZ02</td> </tr><tr bgcolor=""> <th>CA_IND</th> <td>N</td> </tr><tr bgcolor="#E3E3F3"> <th>PLN_AREA_N</th> <td>BUKIT MERAH</td> </tr><tr bgcolor=""> <th>PLN_AREA_C</th> <td>BM</td> </tr><tr bgcolor="#E3E3F3"> <th>REGION_N</th> <td>CENTRAL REGION</td> </tr><tr bgcolor=""> <th>REGION_C</th> <td>CR</td> </tr><tr bgcolor="#E3E3F3"> <th>INC_CRC</th> <td>085EF219A5A1AEAD</td> </tr><tr bgcolor=""> <th>FMEL_UPD_D</th> <td>20191223152313</td> </tr></table></center>


,Name,Description,geometry
0,kml_1,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.81454 1.28239 0, 103.81774 1.2..."
1,kml_2,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.82209 1.28049 0, 103.8221 1.28..."
2,kml_3,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84375 1.28508 0, 103.844 1.284..."
3,kml_4,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84962 1.28412 0, 103.84955 1.2..."
4,kml_5,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.85253 1.28617 0, 103.85253 1.2..."
